# Pipeline ETL — Censo da Educação Superior (INEP) 2022–2024

**Projeto de Integração · Grupo 6 · Responsável: Pessoa 1 (ETL)**

Este notebook implementa o caminho **ETL (Extract, Transform, Load)** do projeto. No ETL, os dados são transformados **antes** de chegar ao banco: a limpeza, a harmonização entre os anos e a montagem do esquema estrela acontecem em memória, com **pandas**. Só o resultado final, já modelado, é carregado no **PostgreSQL**.

As decisões de modelagem seguem o relatório de perfilamento do grupo, [`docs/relatórios/perfilamento.md`](../docs/relatórios/perfilamento.md), citado ao longo do notebook como *perfilamento §N*. O mesmo esquema estrela é construído pelo caminho ELT (carga bruta + dbt), e os dois resultados são comparados no final.

**Etapas**

0. Configuração do ambiente
1. Extração dos microdados de 2022, 2023 e 2024
2. Transformação
3. Modelagem do esquema estrela
4. Carga no PostgreSQL

## 0. Configuração do ambiente

**No computador:** o notebook usa o ambiente virtual `.venv` do repositório, com as bibliotecas de `requirements.txt` (`pip install -r requirements.txt`). Selecione o kernel `.venv` no VS Code.

**No Google Colab:** a célula abaixo detecta o Colab e instala o que falta (o Colab já vem com pandas).

Bibliotecas usadas:

- **pandas:** leitura e transformação dos dados;
- **SQLAlchemy** e **psycopg2:** conexão com o PostgreSQL, usada na etapa de carga.

In [1]:
import subprocess
import sys
from pathlib import Path

EM_COLAB = "google.colab" in sys.modules
if EM_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "sqlalchemy==2.1.8", "psycopg2-binary==2.9.13"], check=True)

import numpy as np
import pandas as pd
import sqlalchemy

print("Python:    ", sys.version.split()[0])
print("pandas:    ", pd.__version__)
print("SQLAlchemy:", sqlalchemy.__version__)

Python:     3.12.10
pandas:     3.1.1
SQLAlchemy: 2.1.8


### Caminhos dos arquivos

Os microdados do INEP são grandes (de 332 MB a 453 MB por ano, só o arquivo de cursos) e por isso **não ficam no GitHub**: a pasta `data/raw/` está no `.gitignore`. Cada integrante baixa os arquivos no [site do INEP](https://www.gov.br/inep/pt-br/acesso-a-informacao/dados-abertos/microdados/censo-da-educacao-superior) e copia os CSVs da pasta `dados` de cada ano para `data/raw/<ano>/`, a mesma estrutura usada no notebook de perfilamento.

Usamos dois arquivos de cada ano:

- `MICRODADOS_CADASTRO_CURSOS_<ano>.CSV`: cursos, com vagas, inscritos, ingressantes, matriculados e concluintes;
- `MICRODADOS_ED_SUP_IES_<ano>.CSV`: cadastro das instituições de ensino superior (IES).

No Colab, a célula monta o Google Drive, onde a pasta `data/raw` deve estar.

In [2]:
if EM_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PASTA_DADOS = Path("/content/drive/MyDrive/Projeto-Integra-o-Grupo-6---BD/data/raw")  # ajuste se necessário
else:
    PASTA_DADOS = Path("..") / "data" / "raw"  # este notebook fica em notebooks/

ANOS = [2022, 2023, 2024]
ARQUIVOS = {
    ano: {
        "cursos": PASTA_DADOS / str(ano) / f"MICRODADOS_CADASTRO_CURSOS_{ano}.CSV",
        "ies": PASTA_DADOS / str(ano) / f"MICRODADOS_ED_SUP_IES_{ano}.CSV",
    }
    for ano in ANOS
}

for ano, arquivos in ARQUIVOS.items():
    for tipo, caminho in arquivos.items():
        tamanho = f"{caminho.stat().st_size / 1e6:,.0f} MB" if caminho.exists() else "NÃO ENCONTRADO"
        print(f"{ano} | {tipo:<6} | {tamanho}")

2022 | cursos | 332 MB
2022 | ies    | 1 MB
2023 | cursos | 393 MB
2023 | ies    | 1 MB
2024 | cursos | 453 MB
2024 | ies    | 1 MB


## 1. Extração

### 1.1 Codificação dos arquivos

O `pd.read_csv` usa a codificação **UTF-8** por padrão, mas os microdados do INEP estão em **ISO-8859-1 (latin1)** (perfilamento §2). A célula abaixo mostra o erro que aparece quando a codificação não é informada.

In [3]:
try:
    pd.read_csv(ARQUIVOS[2022]["ies"], sep=";", nrows=5)
except UnicodeDecodeError as erro:
    print("A leitura com UTF-8 falhou:")
    print(erro)

A leitura com UTF-8 falhou:
'utf-8' codec can't decode byte 0xed in position 4: unexpected end of data


O byte indicado na mensagem é um caractere acentuado em latin1 (por exemplo, `0xed` é a letra **í** e `0xe1` é a letra **á**), que não é válido em UTF-8. Por isso todas as leituras usam `encoding="latin1"`, com o separador ponto e vírgula (`sep=";"`).

### 1.2 Colunas selecionadas

O arquivo de cursos tem de 200 a 223 colunas, conforme o ano. Lemos só as necessárias para o esquema estrela:

- as **24 colunas** verificadas no perfilamento (§7), que existem nos três anos;
- as **vagas e inscritos** (`QT_VG_TOTAL`, `QT_INSCRITO_TOTAL`), que entram na fato porque as linhas de EAD em nível Brasil só têm essas medidas (§9);
- as colunas da **reserva de vagas étnica**, que mudam em 2024 (§8);
- os **nomes e atributos descritivos** das dimensões (município, UF, curso, CINE, IES).

A mantenedora fica fora do modelo: é o atributo mais instável entre os anos e nenhuma análise usa (§13).

`QT_ING_RVETNICO` só existe até 2023, e `QT_ING_RVPPI`, `QT_ING_RVQUILO` e `QT_ING_RVPOVT` só existem em 2024. A leitura carrega as colunas que existirem em cada ano, e a diferença é tratada na transformação.

In [4]:
COLUNAS_CURSOS = [
    # Identificação, tempo e tipo de oferta
    "NU_ANO_CENSO", "CO_IES", "CO_CURSO", "NO_CURSO", "TP_DIMENSAO",
    # Local de oferta
    "CO_REGIAO", "NO_REGIAO", "CO_UF", "SG_UF", "NO_UF",
    "CO_MUNICIPIO", "NO_MUNICIPIO", "IN_CAPITAL",
    # Características do curso
    "TP_GRAU_ACADEMICO", "TP_MODALIDADE_ENSINO", "TP_NIVEL_ACADEMICO", "IN_GRATUITO",
    # Área do conhecimento (CINE)
    "CO_CINE_ROTULO", "NO_CINE_ROTULO",
    "NO_CINE_AREA_GERAL", "NO_CINE_AREA_ESPECIFICA", "NO_CINE_AREA_DETALHADA",
    # Medidas
    "QT_VG_TOTAL", "QT_INSCRITO_TOTAL", "QT_ING", "QT_MAT", "QT_CONC",
    "QT_ING_FIES", "QT_ING_PROUNII", "QT_ING_PROUNIP",
    "QT_ING_PRETA", "QT_ING_PARDA", "QT_ING_INDIGENA",
    "QT_ING_RESERVA_VAGA",
    # Reserva étnica: coluna única até 2023, três colunas em 2024
    "QT_ING_RVETNICO", "QT_ING_RVPPI", "QT_ING_RVQUILO", "QT_ING_RVPOVT",
]

COLUNAS_IES = [
    "NU_ANO_CENSO", "CO_IES", "NO_IES", "SG_IES",
    "TP_ORGANIZACAO_ACADEMICA", "TP_CATEGORIA_ADMINISTRATIVA", "TP_REDE",
    "IN_COMUNITARIA", "IN_CONFESSIONAL",
    "NO_MUNICIPIO_IES", "SG_UF_IES", "NO_REGIAO_IES",
]

print(f"{len(COLUNAS_CURSOS)} colunas de cursos e {len(COLUNAS_IES)} colunas de IES selecionadas")

38 colunas de cursos e 12 colunas de IES selecionadas


### 1.3 Leitura dos três anos

A função `ler_csv` aplica as configurações de leitura e faz três escolhas para economizar memória (só o arquivo de 2024 tem 720 mil linhas):

- **`usecols` com uma função**, em vez da lista direta: o pandas carrega as colunas da lista que existirem no arquivo, sem erro quando uma delas não existe naquele ano;
- **leitura em blocos** (`chunksize`) de 100 mil linhas, juntados no final com `pd.concat`, para evitar um pico de memória;
- **colunas de texto** (que começam com `NO_` e `SG_`) lidas como texto e convertidas para `category`, que guarda cada valor repetido (como o nome de um município) uma única vez. Ler essas colunas como texto também evita o `DtypeWarning` de tipos mistos em `NO_MUNICIPIO` (perfilamento §2).

Ao final, um resumo mostra o tamanho de cada arquivo e as colunas da lista que **não** existem em cada ano.

In [5]:
def ler_csv(caminho, colunas):
    """Lê um CSV do INEP em blocos, mantendo apenas as colunas selecionadas que existirem no arquivo."""
    colunas_texto = [c for c in colunas if c.startswith(("NO_", "SG_"))]
    blocos = pd.read_csv(
        caminho,
        sep=";",
        encoding="latin1",
        usecols=lambda coluna: coluna in colunas,
        dtype={c: "string" for c in colunas_texto},
        chunksize=100_000,
    )
    df = pd.concat(blocos, ignore_index=True)
    texto = [c for c in colunas_texto if c in df.columns]
    df[texto] = df[texto].astype("category")
    return df


dfs_cursos = {}
dfs_ies = {}
resumo = []

for ano, arquivos in ARQUIVOS.items():
    dfs_cursos[ano] = ler_csv(arquivos["cursos"], COLUNAS_CURSOS)
    dfs_ies[ano] = ler_csv(arquivos["ies"], COLUNAS_IES)
    resumo.append({
        "ano": ano,
        "linhas_cursos": len(dfs_cursos[ano]),
        "colunas_cursos": dfs_cursos[ano].shape[1],
        "ausentes_cursos": [c for c in COLUNAS_CURSOS if c not in dfs_cursos[ano].columns],
        "linhas_ies": len(dfs_ies[ano]),
        "colunas_ies": dfs_ies[ano].shape[1],
        "ausentes_ies": [c for c in COLUNAS_IES if c not in dfs_ies[ano].columns],
    })

pd.set_option("display.max_colwidth", None)
display(pd.DataFrame(resumo).set_index("ano"))

,linhas_cursos,colunas_cursos,ausentes_cursos,linhas_ies,colunas_ies,ausentes_ies
ano,,,,,,
2022,573019,35,"[QT_ING_RVPPI, QT_ING_RVQUILO, QT_ING_RVPOVT]",2595,9,"[TP_REDE, IN_COMUNITARIA, IN_CONFESSIONAL]"
2023,671610,35,"[QT_ING_RVPPI, QT_ING_RVQUILO, QT_ING_RVPOVT]",2580,12,[]
2024,720349,37,[QT_ING_RVETNICO],2561,12,[]


As ausências confirmam o perfilamento (§3 e §4): a reserva étnica muda de formato em 2024, e o cadastro de IES de 2022 não tem `TP_REDE`, `IN_COMUNITARIA` e `IN_CONFESSIONAL`.

### 1.4 Primeira olhada nos dados

In [6]:
display(dfs_cursos[2024].head())
dfs_cursos[2024].info()

,NU_ANO_CENSO,NO_REGIAO,CO_REGIAO,NO_UF,SG_UF,CO_UF,NO_MUNICIPIO,CO_MUNICIPIO,IN_CAPITAL,TP_DIMENSAO,...,QT_ING_INDIGENA,QT_MAT,QT_CONC,QT_ING_FIES,QT_ING_PROUNII,QT_ING_PROUNIP,QT_ING_RESERVA_VAGA,QT_ING_RVPPI,QT_ING_RVQUILO,QT_ING_RVPOVT
0,2024,<NA>,NaN,<NA>,<NA>,NaN,<NA>,NaN,NaN,2,...,0,17,0,0,0,0,0,0,0,0
1,2024,<NA>,NaN,<NA>,<NA>,NaN,<NA>,NaN,NaN,2,...,0,0,0,0,0,0,0,0,0,0
2,2024,<NA>,NaN,<NA>,<NA>,NaN,<NA>,NaN,NaN,2,...,0,0,0,0,0,0,0,0,0,0
3,2024,<NA>,NaN,<NA>,<NA>,NaN,<NA>,NaN,NaN,2,...,0,0,0,0,0,0,0,0,0,0
4,2024,<NA>,NaN,<NA>,<NA>,NaN,<NA>,NaN,NaN,2,...,0,24,0,0,0,0,0,0,0,0


<class 'pandas.DataFrame'>
RangeIndex: 720349 entries, 0 to 720348
Data columns (total 37 columns):
 #   Column                   Non-Null Count   Dtype   
---  ------                   --------------   -----   
 0   NU_ANO_CENSO             720349 non-null  int64   
 1   NO_REGIAO                708571 non-null  category
 2   CO_REGIAO                708571 non-null  float64 
 3   NO_UF                    708571 non-null  category
 4   SG_UF                    708571 non-null  category
 5   CO_UF                    708571 non-null  float64 
 6   NO_MUNICIPIO             708571 non-null  category
 7   CO_MUNICIPIO             708571 non-null  float64 
 8   IN_CAPITAL               708571 non-null  float64 
 9   TP_DIMENSAO              720349 non-null  int64   
 10  CO_IES                   720349 non-null  int64   
 11  NO_CURSO                 720349 non-null  category
 12  CO_CURSO                 720349 non-null  int64   
 13  NO_CINE_ROTULO           720349 non-null  category
 14 

In [7]:
display(dfs_ies[2024].head())

,NU_ANO_CENSO,NO_REGIAO_IES,SG_UF_IES,NO_MUNICIPIO_IES,TP_ORGANIZACAO_ACADEMICA,TP_REDE,TP_CATEGORIA_ADMINISTRATIVA,IN_COMUNITARIA,IN_CONFESSIONAL,CO_IES,NO_IES,SG_IES
0,2024,Centro-Oeste,MT,Cuiabá,1,1,1,0,0,1,UNIVERSIDADE FEDERAL DE MATO GROSSO,UFMT
1,2024,Centro-Oeste,DF,Brasília,1,1,1,0,0,2,UNIVERSIDADE DE BRASÍLIA,UNB
2,2024,Nordeste,SE,São Cristóvão,1,1,1,0,0,3,UNIVERSIDADE FEDERAL DE SERGIPE,UFS
3,2024,Norte,AM,Manaus,1,1,1,0,0,4,UNIVERSIDADE FEDERAL DO AMAZONAS,UFAM
4,2024,Nordeste,PI,Teresina,1,1,1,0,0,5,UNIVERSIDADE FEDERAL DO PIAUÍ,UFPI


### 1.5 Conferência da leitura

Comparamos a leitura com os valores de referência do perfilamento (§15): número de linhas, cursos distintos e totais de ingressantes e matrículas. Essa tabela também será a referência da conferência entre ETL e ELT.

In [8]:
REFERENCIA = pd.DataFrame({
    "linhas": [573_019, 671_610, 720_349],
    "cursos distintos": [45_296, 46_317, 46_150],
    "QT_ING": [4_756_957, 4_994_192, 5_010_613],
    "QT_MAT": [9_444_116, 9_977_217, 10_227_266],
}, index=pd.Index(ANOS, name="ano"))

lido = pd.DataFrame({
    "linhas": {ano: len(df) for ano, df in dfs_cursos.items()},
    "cursos distintos": {ano: df["CO_CURSO"].nunique() for ano, df in dfs_cursos.items()},
    "QT_ING": {ano: df["QT_ING"].sum() for ano, df in dfs_cursos.items()},
    "QT_MAT": {ano: df["QT_MAT"].sum() for ano, df in dfs_cursos.items()},
})

conferencia_leitura = lido.merge(REFERENCIA, left_index=True, right_index=True, suffixes=("", " (referência)"))
conferencia_leitura["confere"] = (lido == REFERENCIA).all(axis=1)
display(conferencia_leitura)

,linhas,cursos distintos,QT_ING,QT_MAT,linhas (referência),cursos distintos (referência),QT_ING (referência),QT_MAT (referência),confere
2022,573019,45296,4756957,9444116,573019,45296,4756957,9444116,True
2023,671610,46317,4994192,9977217,671610,46317,4994192,9977217,True
2024,720349,46150,5010613,10227266,720349,46150,5010613,10227266,True


## 2. Transformação

Nesta etapa os três anos viram uma base única e consistente. Tudo acontece em memória, com pandas, **antes** de qualquer dado chegar ao banco. É isso que caracteriza o ETL.

| Passo | O que resolve | Perfilamento |
| --- | --- | --- |
| 2.1 Juntar os anos | Empilha 2022, 2023 e 2024 numa tabela de cursos e numa de IES | — |
| 2.2 Local de oferta | Substitui o código 0 (2022) e o município nulo (2023–2024) por membros especiais, definidos por `TP_DIMENSAO` | §10 |
| 2.3 Rede de ensino | Deriva a rede da categoria administrativa nos três anos e valida contra `TP_REDE` | §12 |
| 2.4 Reserva étnica | Cria `QT_ING_RESERVA_ETNICA`, comparável nos três anos | §8 |
| 2.5 Limpeza de valores | Remove aspas do código CINE e trata o grau acadêmico vazio | §5 |
| 2.6 Decodificação | Troca códigos numéricos por descrições, como 1 → "Presencial" | §5 |
| 2.7 Conferência | Confirma que nenhum ingressante ou matrícula foi perdido ou criado | §15 |

### 2.1 Juntando os três anos

Empilhamos os DataFrames dos três anos com `pd.concat`. Antes disso, igualamos as categorias das colunas de texto entre os anos com `union_categoricals` (que junta, por exemplo, a lista de municípios dos três anos). Sem isso, o pandas converteria as colunas `category` de volta para texto comum, e o uso de memória aumentaria bastante.

Colunas que não existem num ano ficam vazias nas linhas daquele ano. Elas são tratadas nos próximos passos.

In [9]:
from pandas.api.types import union_categoricals


def juntar_anos(dfs):
    """Empilha os DataFrames de cada ano, mantendo as colunas de texto como category."""
    dfs = list(dfs)
    colunas_categoricas = {
        c for df in dfs for c in df.columns if isinstance(df[c].dtype, pd.CategoricalDtype)
    }
    for coluna in colunas_categoricas:
        categorias = union_categoricals([df[coluna] for df in dfs if coluna in df.columns]).categories
        for df in dfs:
            if coluna in df.columns:
                df[coluna] = df[coluna].cat.set_categories(categorias)
    return pd.concat(dfs, ignore_index=True)


cursos = juntar_anos(dfs_cursos.values())
ies = juntar_anos(dfs_ies.values())

print(f"Cursos: {len(cursos):,} linhas = " + " + ".join(f"{len(df):,}" for df in dfs_cursos.values()))
print(f"IES:    {len(ies):,} linhas = " + " + ".join(f"{len(df):,}" for df in dfs_ies.values()))

Cursos: 1,964,978 linhas = 573,019 + 671,610 + 720,349
IES:    7,736 linhas = 2,595 + 2,580 + 2,561


### 2.2 Local de oferta sem município

O dicionário do INEP informa que cursos a distância não podem ser quantificados por município. Nos dados, essas linhas aparecem de dois jeitos (perfilamento §10):

- **2022:** código `0` em `CO_MUNICIPIO`, `CO_UF` e `CO_REGIAO`, com o nome vazio;
- **2023 e 2024:** colunas de localização vazias (nulas).

A tabela abaixo cruza o tipo de oferta (`TP_DIMENSAO`) com a situação do município em cada ano.

| `TP_DIMENSAO` | Significado |
| --- | --- |
| 1 | Curso presencial no Brasil |
| 2 | Curso a distância no Brasil (uma linha por polo) |
| 3 | Curso a distância com dados só em nível Brasil |
| 4 | Curso a distância de instituição brasileira no exterior |

In [10]:
situacao_municipio = pd.Series(
    np.select([cursos["CO_MUNICIPIO"].isna(), cursos["CO_MUNICIPIO"] == 0], ["nulo", "código 0"], "válido"),
    index=cursos.index, name="município",
)
print("ANTES: situação do município por ano e TP_DIMENSAO")
display(pd.crosstab([cursos["NU_ANO_CENSO"], cursos["TP_DIMENSAO"]], situacao_municipio))

ANTES: situação do município por ano e TP_DIMENSAO


município                 código 0   nulo  válido
NU_ANO_CENSO TP_DIMENSAO                         
2022         1                   0      0   36089
             2                   1      0  527327
             3                9207      0       0
             4                 395      0       0
2023         1                   0      0   35739
             2                   0      4  624832
             3                   0  10576       0
             4                   0    459       0
2024         1                   0      0   34824
             2                   0      9  673747
             3                   0  11319       0
             4                   0    450       0

As linhas sem município **não são descartadas**: elas carregam ingressantes, matrículas e quase todas as vagas de EAD. Seguindo o perfilamento (§10), elas recebem **membros especiais** na dimensão de localidade, definidos pelo `TP_DIMENSAO`, e não pelo código:

| Código especial | Regra | Município na dimensão |
| --- | --- | --- |
| -1 | `TP_DIMENSAO` = 3, sem município | Sem município (EAD nível Brasil) |
| -2 | `TP_DIMENSAO` = 4 | Exterior |
| -3 | qualquer outra linha sem município (poucas linhas de `TP_DIMENSAO` = 2) | Não informado |

O perfilamento não define o caso das poucas linhas de `TP_DIMENSAO` = 2 sem município; elas recebem o membro "Não informado".

In [11]:
sem_municipio = cursos["CO_MUNICIPIO"].isna() | (cursos["CO_MUNICIPIO"] == 0)

codigo_especial = pd.Series(np.select(
    [cursos["TP_DIMENSAO"] == 4, sem_municipio & (cursos["TP_DIMENSAO"] == 3), sem_municipio],
    [-2, -1, -3],
    default=0,
), index=cursos.index)

# Nome do município, nome da UF, sigla da UF e região de cada membro especial
MEMBROS_ESPECIAIS = {
    -1: ("Sem município (EAD nível Brasil)", "Não se aplica", "BR", "Não se aplica"),
    -2: ("Exterior", "Exterior", "EX", "Exterior"),
    -3: ("Não informado", "Não informado", "NI", "Não informado"),
}
COLUNAS_TEXTO_LOCAL = ["NO_MUNICIPIO", "NO_UF", "SG_UF", "NO_REGIAO"]

for codigo, valores in MEMBROS_ESPECIAIS.items():
    linhas = codigo_especial == codigo
    cursos.loc[linhas, ["CO_MUNICIPIO", "CO_UF", "CO_REGIAO", "IN_CAPITAL"]] = [codigo, codigo, codigo, -1]
    for coluna, valor in zip(COLUNAS_TEXTO_LOCAL, valores):
        if valor not in cursos[coluna].cat.categories:
            cursos[coluna] = cursos[coluna].cat.add_categories(valor)
        cursos.loc[linhas, coluna] = valor

COLUNAS_CODIGO_LOCAL = ["CO_MUNICIPIO", "CO_UF", "CO_REGIAO", "IN_CAPITAL"]
assert cursos[COLUNAS_CODIGO_LOCAL + COLUNAS_TEXTO_LOCAL].notna().all().all(), "ainda há localização vazia"
cursos[COLUNAS_CODIGO_LOCAL] = cursos[COLUNAS_CODIGO_LOCAL].astype("int64")

print("DEPOIS: linhas por membro especial de localidade")
display(pd.crosstab(
    cursos["NU_ANO_CENSO"],
    cursos["CO_MUNICIPIO"].where(cursos["CO_MUNICIPIO"] < 0).map({c: v[0] for c, v in MEMBROS_ESPECIAIS.items()}),
))

DEPOIS: linhas por membro especial de localidade


CO_MUNICIPIO,Exterior,Não informado,Sem município (EAD nível Brasil)
NU_ANO_CENSO,,,
2022,395,1,9207
2023,459,4,10576
2024,450,9,11319


### 2.3 Rede de ensino (pública ou privada)

O cadastro de IES de 2022 não tem `TP_REDE`. O perfilamento (§12) mostrou que a rede é determinada sem exceções pela **categoria administrativa**, que existe nos três anos:

| Categoria (`TP_CATEGORIA_ADMINISTRATIVA`) | Rede |
| --- | --- |
| 1 Pública Federal, 2 Pública Estadual, 3 Pública Municipal, 7 Especial | Pública |
| 4 Privada com fins lucrativos, 5 Privada sem fins lucrativos | Privada |

Por isso a rede é **derivada da categoria em todos os anos**, e a `TP_REDE` original serve de validação em 2023 e 2024. A célula para com erro se aparecer uma categoria fora da tabela (as categorias 6, 8 e 9 existem no dicionário, mas não nos dados).

In [12]:
REDE_POR_CATEGORIA = {1: 1, 2: 1, 3: 1, 7: 1, 4: 2, 5: 2}

categorias_inesperadas = set(ies["TP_CATEGORIA_ADMINISTRATIVA"]) - set(REDE_POR_CATEGORIA)
assert not categorias_inesperadas, f"categoria administrativa sem regra de rede: {categorias_inesperadas}"

rede_derivada = ies["TP_CATEGORIA_ADMINISTRATIVA"].map(REDE_POR_CATEGORIA)

tem_rede = ies["TP_REDE"].notna()
divergencias = (rede_derivada[tem_rede] != ies.loc[tem_rede, "TP_REDE"]).sum()
print(f"Validação em 2023 e 2024: {tem_rede.sum():,} IES comparadas, {divergencias} divergências")
assert divergencias == 0

ies["TP_REDE"] = rede_derivada.astype("int64")
display(pd.crosstab([ies["NU_ANO_CENSO"]], ies["TP_REDE"], colnames=["rede (1 = pública, 2 = privada)"]))

Validação em 2023 e 2024: 5,141 IES comparadas, 0 divergências


"rede (1 = pública, 2 = privada)",1,2
NU_ANO_CENSO,,
2022,312,2283
2023,316,2264
2024,317,2244


`IN_COMUNITARIA` e `IN_CONFESSIONAL` também não existem em 2022 e não podem ser deduzidas. Nas IES de 2022 elas recebem `-1`, que será decodificado como "Não informado".

In [13]:
ies[["IN_COMUNITARIA", "IN_CONFESSIONAL"]] = ies[["IN_COMUNITARIA", "IN_CONFESSIONAL"]].fillna(-1).astype("int64")
display(ies.groupby("NU_ANO_CENSO")[["IN_COMUNITARIA", "IN_CONFESSIONAL"]].agg(lambda s: (s == -1).sum()))

,IN_COMUNITARIA,IN_CONFESSIONAL
NU_ANO_CENSO,,
2022,2595,2595
2023,0,0
2024,0,0


### 2.4 Reserva de vagas de cunho étnico

Até 2023 o INEP informava os ingressantes por reserva étnica numa só coluna, `QT_ING_RVETNICO`. Em 2024 ela deixou de existir, e a informação passou a ser detalhada por grupo (perfilamento §3 e §8).

Seguindo o perfilamento, criamos a medida **`QT_ING_RESERVA_ETNICA`**:

- **2022 e 2023:** valor de `QT_ING_RVETNICO`;
- **2024:** soma de `QT_ING_RVPPI` (pretos, pardos e indígenas), `QT_ING_RVQUILO` (quilombolas) e `QT_ING_RVPOVT` (povos e comunidades tradicionais).

> **Aproximação.** A série fica suave, sem salto em 2024, o que torna plausível a substituição. O dicionário não confirma a equivalência, e há risco de dupla contagem se um aluno constar em mais de um grupo (perfilamento §8).

In [14]:
COLUNAS_RESERVA_2024 = ["QT_ING_RVPPI", "QT_ING_RVQUILO", "QT_ING_RVPOVT"]
eh_2024 = cursos["NU_ANO_CENSO"] == 2024

cursos["QT_ING_RESERVA_ETNICA"] = cursos["QT_ING_RVETNICO"]
cursos.loc[eh_2024, "QT_ING_RESERVA_ETNICA"] = cursos.loc[eh_2024, COLUNAS_RESERVA_2024].sum(axis=1)
cursos["QT_ING_RESERVA_ETNICA"] = cursos["QT_ING_RESERVA_ETNICA"].astype("int64")

reserva = cursos.groupby("NU_ANO_CENSO")[["QT_ING", "QT_ING_RVETNICO"] + COLUNAS_RESERVA_2024 + ["QT_ING_RESERVA_ETNICA"]].sum()
reserva["% dos ingressantes"] = (reserva["QT_ING_RESERVA_ETNICA"] / reserva["QT_ING"] * 100).round(2)
display(reserva)

# As colunas de origem já estão representadas na nova medida
cursos = cursos.drop(columns=["QT_ING_RVETNICO"] + COLUNAS_RESERVA_2024)

,QT_ING,QT_ING_RVETNICO,QT_ING_RVPPI,QT_ING_RVQUILO,QT_ING_RVPOVT,QT_ING_RESERVA_ETNICA,% dos ingressantes
NU_ANO_CENSO,,,,,,,
2022,4756957,63925.0,0.0,0.0,0.0,63925,1.34
2023,4994192,72270.0,0.0,0.0,0.0,72270,1.45
2024,5010613,0.0,79969.0,2314.0,758.0,83041,1.66


Na tabela, uma coluna aparece com 0 num ano quando ela não existe naquele ano (o pandas soma vazios como zero). A coluna `QT_ING_RESERVA_ETNICA` é a única comparável entre os três anos, e os percentuais reproduzem os do perfilamento (1,34%, 1,45% e 1,66%).

### 2.5 Limpeza de valores

- **`CO_CINE_ROTULO` com aspas:** o código vem como `"0011A09"`, com as aspas dentro do valor. Removemos as aspas para usar o código como chave.
- **`TP_GRAU_ACADEMICO` vazio:** o dicionário define o vazio como "não aplicável" (perfilamento §5), caso dos cursos de Área Básica de Ingresso (ABI) e dos cursos sequenciais. Usamos o código `0` para esse caso.

In [15]:
print("CO_CINE_ROTULO antes: ", cursos["CO_CINE_ROTULO"].drop_duplicates().head(3).tolist())
cursos["CO_CINE_ROTULO"] = cursos["CO_CINE_ROTULO"].astype("string").str.strip('"').astype("category")
print("CO_CINE_ROTULO depois:", cursos["CO_CINE_ROTULO"].drop_duplicates().head(3).tolist())

print(f"\nTP_GRAU_ACADEMICO vazio antes:  {cursos['TP_GRAU_ACADEMICO'].isna().sum():,}")
cursos["TP_GRAU_ACADEMICO"] = cursos["TP_GRAU_ACADEMICO"].fillna(0).astype("int64")
print(f"TP_GRAU_ACADEMICO vazio depois: {cursos['TP_GRAU_ACADEMICO'].isna().sum():,}")

CO_CINE_ROTULO antes:  ['"0011A09"', '"0413A02"', '"0588P01"']
CO_CINE_ROTULO depois: ['0011A09', '0413A02', '0588P01']

TP_GRAU_ACADEMICO vazio antes:  3,766
TP_GRAU_ACADEMICO vazio depois: 0


### 2.6 Decodificação dos códigos

Para que o Data Warehouse seja legível em qualquer consulta, cada código ganha uma coluna de descrição (`DS_...`), com os textos do dicionário de dados do INEP. O perfilamento (§5) confirmou que as listas de códigos são **idênticas nos três anos**, então um único dicionário serve para todos. Os códigos `-1` e `0` criados nos passos anteriores também ganham descrição.

A função `decodificar` para com erro se algum código ficar sem descrição.

In [16]:
SIM_NAO = {0: "Não", 1: "Sim"}
SIM_NAO_NI = {-1: "Não informado", 0: "Não", 1: "Sim"}

MAPA_DIMENSAO = {
    1: "Presencial no Brasil",
    2: "EAD no Brasil",
    3: "EAD com dados só em nível Brasil",
    4: "EAD de instituição brasileira no exterior",
}

MAPAS_CURSOS = {
    "TP_DIMENSAO": MAPA_DIMENSAO,
    "TP_MODALIDADE_ENSINO": {1: "Presencial", 2: "EAD"},
    "TP_GRAU_ACADEMICO": {0: "Não aplicável", 1: "Bacharelado", 2: "Licenciatura",
                          3: "Tecnológico", 4: "Bacharelado e Licenciatura"},
    "TP_NIVEL_ACADEMICO": {1: "Graduação", 2: "Sequencial de Formação Específica"},
    "IN_GRATUITO": SIM_NAO,
    "IN_CAPITAL": {-1: "Não se aplica", 0: "Não", 1: "Sim"},
}

MAPAS_IES = {
    "TP_REDE": {1: "Pública", 2: "Privada"},
    "TP_CATEGORIA_ADMINISTRATIVA": {1: "Pública Federal", 2: "Pública Estadual", 3: "Pública Municipal",
                                    4: "Privada com fins lucrativos", 5: "Privada sem fins lucrativos",
                                    7: "Especial"},
    "TP_ORGANIZACAO_ACADEMICA": {1: "Universidade", 2: "Centro Universitário", 3: "Faculdade",
                                 4: "Instituto Federal", 5: "Centro Federal de Educação Tecnológica"},
    "IN_COMUNITARIA": SIM_NAO_NI,
    "IN_CONFESSIONAL": SIM_NAO_NI,
}


def decodificar(df, mapas):
    """Cria uma coluna DS_<nome> com a descrição de cada código e devolve a correspondência usada."""
    resumo = {}
    for coluna, mapa in mapas.items():
        nova = "DS_" + coluna[3:]  # TP_REDE -> DS_REDE, IN_GRATUITO -> DS_GRATUITO
        df[nova] = df[coluna].map(mapa).astype("category")
        sem_descricao = df[nova].isna().sum()
        assert sem_descricao == 0, f"{coluna}: {sem_descricao} linhas com código sem descrição"
        resumo[coluna] = df.groupby([coluna, nova], observed=True).size()
    return pd.concat(resumo, names=["coluna", "código", "descrição"]).rename("linhas").to_frame()


print("Cursos:")
display(decodificar(cursos, MAPAS_CURSOS))
print("IES:")
display(decodificar(ies, MAPAS_IES))

Cursos:


linhas
coluna               código descrição                                         
TP_DIMENSAO           1     Presencial no Brasil                        106652
                      2     EAD no Brasil                              1825920
                      3     EAD com dados só em nível Brasil             31102
                      4     EAD de instituição brasileira no exterior     1304
TP_MODALIDADE_ENSINO  1     Presencial                                  106652
                      2     EAD                                        1858326
TP_GRAU_ACADEMICO     0     Não aplicável                                 5173
                      1     Bacharelado                                 647767
                      2     Licenciatura                                384740
                      3     Tecnológico                                 927298
TP_NIVEL_ACADEMICO    1     Graduação                                  1964960
                      2     Sequencial de Formação Específica               18
IN_GRATUITO           0     Não                                        1910263
                      1     Sim                                          54715
IN_CAPITAL           -1     Não se aplica                                32420
                      0     Não                                        1725566
                      1     Sim                                         206992

IES:


linhas
coluna                      código descrição                                     
TP_REDE                      1     Pública                                    945
                             2     Privada                                   6791
TP_CATEGORIA_ADMINISTRATIVA  1     Pública Federal                            363
                             2     Pública Estadual                           410
                             3     Pública Municipal                           92
                             4     Privada com fins lucrativos               4408
                             5     Privada sem fins lucrativos               2383
                             7     Especial                                    80
TP_ORGANIZACAO_ACADEMICA     1     Universidade                               616
                             2     Centro Universitário                      1191
                             3     Faculdade                                 5806
                             4     Instituto Federal                          117
                             5     Centro Federal de Educação Tecnológica       6
IN_COMUNITARIA              -1     Não informado                             2595
                             0     Não                                       4975
                             1     Sim                                        166
IN_CONFESSIONAL             -1     Não informado                             2595
                             0     Não                                       5031
                             1     Sim                                        110

A tabela confirma outro achado do perfilamento (§9): a modalidade é determinada pelo `TP_DIMENSAO` (a dimensão 1 é toda presencial, e as dimensões 2, 3 e 4 são todas EAD).

### 2.7 Conferência da transformação

Nenhum passo da transformação deveria criar ou perder alunos ou vagas. Comparamos, por ano, as somas da base transformada com a referência da seção 1.5 e com as vagas lidas originalmente.

Depois da conferência, apagamos os DataFrames separados por ano para liberar memória para a modelagem.

In [17]:
transformado = cursos.groupby("NU_ANO_CENSO").agg(
    linhas=("CO_CURSO", "size"), QT_ING=("QT_ING", "sum"), QT_MAT=("QT_MAT", "sum"), QT_VG_TOTAL=("QT_VG_TOTAL", "sum"),
)
original = pd.DataFrame({
    "linhas": REFERENCIA["linhas"], "QT_ING": REFERENCIA["QT_ING"], "QT_MAT": REFERENCIA["QT_MAT"],
    "QT_VG_TOTAL": {ano: df["QT_VG_TOTAL"].sum() for ano, df in dfs_cursos.items()},
})
conferencia_transformacao = transformado.merge(original, left_index=True, right_index=True, suffixes=("", " (original)"))
conferencia_transformacao["confere"] = (transformado.values == original.loc[transformado.index].values).all(axis=1)
display(conferencia_transformacao)

del dfs_cursos, dfs_ies

,linhas,QT_ING,QT_MAT,QT_VG_TOTAL,linhas (original),QT_ING (original),QT_MAT (original),QT_VG_TOTAL (original),confere
NU_ANO_CENSO,,,,,,,,,
2022,573019,4756957,9444116,22830485,573019,4756957,9444116,22830485,True
2023,671610,4994192,9977217,24621965,671610,4994192,9977217,24621965,True
2024,720349,5010613,10227266,23665419,720349,5010613,10227266,23665419,True


## 3. Modelagem do esquema estrela

Com a base limpa, montamos o esquema estrela definido no perfilamento (§11, §13, §14 e §16): uma tabela **fato** com as medidas, ligada a **dimensões** que descrevem cada medida.

| Tabela | Uma linha por | Chave natural | Por quê |
| --- | --- | --- | --- |
| `fato_oferta_curso` | curso × município × tipo de oferta × ano | `NU_ANO_CENSO` + `CO_CURSO` + `CO_MUNICIPIO` + `TP_DIMENSAO` | Única nos três anos; sem `TP_DIMENSAO` há repetições legítimas (§11) |
| `dim_tempo` | ano | `NU_ANO_CENSO` | — |
| `dim_dimensao` | tipo de oferta | `TP_DIMENSAO` | Varia dentro do mesmo curso, por isso fica fora da `dim_curso` (§11) |
| `dim_curso` | curso **e ano** | `CO_CURSO` + `NU_ANO_CENSO` | Nome, grau e IES mudam entre anos; a fato aponta para a versão do próprio ano (§13) |
| `dim_ies` | IES **e ano** | `CO_IES` + `NU_ANO_CENSO` | Nome, categoria e rede mudam entre anos (§13 e §14) |
| `dim_area_cine` | rótulo CINE | `CO_CINE_ROTULO` | Só um rótulo muda de nome, por renomeação; fica o nome mais recente (§14) |
| `dim_localidade` | município ou membro especial | `CO_MUNICIPIO` | Inclui os membros especiais da seção 2.2 |

Decisões gerais:

- **Surrogate keys:** cada dimensão ganha uma chave própria (`sk_...`), um número sequencial independente dos códigos do INEP. A fato guarda só essas chaves e as medidas.
- **Nomes de colunas em minúsculas:** no PostgreSQL, nomes com maiúsculas exigem aspas em toda consulta. Por isso as colunas do Data Warehouse ficam em minúsculas, com nomes descritivos.

### 3.1 Consistência dentro do ano

A `dim_curso` e a `dim_ies` têm uma linha por código e ano. Isso só funciona se cada curso (e cada IES) tiver **um único valor** de cada atributo dentro de um mesmo ano, como o perfilamento verificou (§14). Repetimos o teste na base transformada: a célula para com erro se algum atributo tiver mais de um valor.

In [18]:
def valores_por_chave(df, chave, atributos):
    """Maior número de valores distintos de cada atributo dentro de uma mesma chave."""
    return df.groupby(chave, observed=True)[atributos].nunique().max()


ATRIBUTOS_CURSO = ["NO_CURSO", "CO_IES", "CO_CINE_ROTULO", "DS_GRAU_ACADEMICO",
                   "DS_MODALIDADE_ENSINO", "DS_NIVEL_ACADEMICO", "DS_GRATUITO"]
ATRIBUTOS_IES = ["NO_IES", "SG_IES", "DS_ORGANIZACAO_ACADEMICA", "DS_CATEGORIA_ADMINISTRATIVA", "DS_REDE"]

consistencia = pd.concat({
    "curso × ano": valores_por_chave(cursos, ["CO_CURSO", "NU_ANO_CENSO"], ATRIBUTOS_CURSO),
    "IES × ano": valores_por_chave(ies, ["CO_IES", "NU_ANO_CENSO"], ATRIBUTOS_IES),
}, names=["chave", "atributo"]).rename("máximo de valores por chave").to_frame()
display(consistencia)
assert (consistencia.iloc[:, 0] <= 1).all(), "atributo com mais de um valor dentro do mesmo ano"

máximo de valores por chave
chave       atributo                                                
curso × ano NO_CURSO                                               1
            CO_IES                                                 1
            CO_CINE_ROTULO                                         1
            DS_GRAU_ACADEMICO                                      1
            DS_MODALIDADE_ENSINO                                   1
            DS_NIVEL_ACADEMICO                                     1
            DS_GRATUITO                                            1
IES × ano   NO_IES                                                 1
            SG_IES                                                 1
            DS_ORGANIZACAO_ACADEMICA                               1
            DS_CATEGORIA_ADMINISTRATIVA                            1
            DS_REDE                                                1

### 3.2 Dimensões

A função `criar_dimensao` ordena as linhas por ano, mantém a ocorrência mais recente de cada chave, renomeia as colunas e cria a surrogate key. Nas dimensões por ano (`dim_curso` e `dim_ies`), o ano faz parte da chave, então cada ano mantém a sua própria versão.

In [19]:
def criar_dimensao(df, colunas, chave, nome_sk):
    """Uma linha por chave, com os atributos do ano mais recente e uma surrogate key sequencial."""
    dimensao = (
        df.sort_values("NU_ANO_CENSO")[list(dict.fromkeys(["NU_ANO_CENSO", *colunas]))]
          .drop_duplicates(subset=chave, keep="last")
          .sort_values(chave)[list(colunas)]
          .rename(columns=colunas)
          .reset_index(drop=True)
    )
    dimensao.insert(0, nome_sk, range(1, len(dimensao) + 1))
    return dimensao


dim_tempo = pd.DataFrame({"ano": ANOS})
dim_tempo.insert(0, "sk_tempo", range(1, len(dim_tempo) + 1))

dim_dimensao = pd.DataFrame({
    "co_dimensao": list(MAPA_DIMENSAO),
    "dimensao_geografica": list(MAPA_DIMENSAO.values()),
    "modalidade": ["Presencial" if codigo == 1 else "EAD" for codigo in MAPA_DIMENSAO],
})
dim_dimensao.insert(0, "sk_dimensao", range(1, len(dim_dimensao) + 1))

dim_curso = criar_dimensao(cursos, {
    "CO_CURSO": "co_curso", "NU_ANO_CENSO": "ano", "NO_CURSO": "nome_curso",
    "DS_GRAU_ACADEMICO": "grau_academico", "DS_MODALIDADE_ENSINO": "modalidade",
    "DS_NIVEL_ACADEMICO": "nivel_academico", "DS_GRATUITO": "gratuito",
}, chave=["CO_CURSO", "NU_ANO_CENSO"], nome_sk="sk_curso")

dim_ies = criar_dimensao(ies, {
    "CO_IES": "co_ies", "NU_ANO_CENSO": "ano", "NO_IES": "nome_ies", "SG_IES": "sigla_ies",
    "DS_ORGANIZACAO_ACADEMICA": "organizacao_academica",
    "DS_CATEGORIA_ADMINISTRATIVA": "categoria_administrativa", "DS_REDE": "rede",
    "DS_COMUNITARIA": "comunitaria", "DS_CONFESSIONAL": "confessional",
    "NO_MUNICIPIO_IES": "municipio_sede", "SG_UF_IES": "uf_sede", "NO_REGIAO_IES": "regiao_sede",
}, chave=["CO_IES", "NU_ANO_CENSO"], nome_sk="sk_ies")

dim_area_cine = criar_dimensao(cursos, {
    "CO_CINE_ROTULO": "co_cine_rotulo", "NO_CINE_ROTULO": "rotulo_cine",
    "NO_CINE_AREA_GERAL": "area_geral", "NO_CINE_AREA_ESPECIFICA": "area_especifica",
    "NO_CINE_AREA_DETALHADA": "area_detalhada",
}, chave=["CO_CINE_ROTULO"], nome_sk="sk_area_cine")

dim_localidade = criar_dimensao(cursos, {
    "CO_MUNICIPIO": "co_municipio", "NO_MUNICIPIO": "municipio", "SG_UF": "sg_uf",
    "NO_UF": "uf", "NO_REGIAO": "regiao", "DS_CAPITAL": "capital",
}, chave=["CO_MUNICIPIO"], nome_sk="sk_localidade")

DIMENSOES = {
    "dim_tempo": dim_tempo, "dim_dimensao": dim_dimensao, "dim_curso": dim_curso,
    "dim_ies": dim_ies, "dim_area_cine": dim_area_cine, "dim_localidade": dim_localidade,
}
display(pd.Series({nome: len(df) for nome, df in DIMENSOES.items()}, name="linhas").to_frame())

,linhas
dim_tempo,3
dim_dimensao,4
dim_curso,137763
dim_ies,7736
dim_area_cine,372
dim_localidade,3681


In [20]:
display(dim_dimensao)
display(dim_localidade[dim_localidade["co_municipio"] < 0])
display(dim_curso.head())
display(dim_ies.head())

,sk_dimensao,co_dimensao,dimensao_geografica,modalidade
0,1,1,Presencial no Brasil,Presencial
1,2,2,EAD no Brasil,EAD
2,3,3,EAD com dados só em nível Brasil,EAD
3,4,4,EAD de instituição brasileira no exterior,EAD


,sk_localidade,co_municipio,municipio,sg_uf,uf,regiao,capital
0,1,-3,Não informado,NI,Não informado,Não informado,Não se aplica
1,2,-2,Exterior,EX,Exterior,Exterior,Não se aplica
2,3,-1,Sem município (EAD nível Brasil),BR,Não se aplica,Não se aplica,Não se aplica


,sk_curso,co_curso,ano,nome_curso,grau_academico,modalidade,nivel_academico,gratuito
0,1,1,2022,Direito,Bacharelado,Presencial,Graduação,Sim
1,2,1,2023,Direito,Bacharelado,Presencial,Graduação,Sim
2,3,1,2024,Direito,Bacharelado,Presencial,Graduação,Sim
3,4,2,2022,Ciências Econômicas,Bacharelado,Presencial,Graduação,Sim
4,5,2,2023,Ciências Econômicas,Bacharelado,Presencial,Graduação,Sim


,sk_ies,co_ies,ano,nome_ies,sigla_ies,organizacao_academica,categoria_administrativa,rede,comunitaria,confessional,municipio_sede,uf_sede,regiao_sede
0,1,1,2022,UNIVERSIDADE FEDERAL DE MATO GROSSO,UFMT,Universidade,Pública Federal,Pública,Não informado,Não informado,Cuiabá,MT,Centro-Oeste
1,2,1,2023,UNIVERSIDADE FEDERAL DE MATO GROSSO,UFMT,Universidade,Pública Federal,Pública,Não,Não,Cuiabá,MT,Centro-Oeste
2,3,1,2024,UNIVERSIDADE FEDERAL DE MATO GROSSO,UFMT,Universidade,Pública Federal,Pública,Não,Não,Cuiabá,MT,Centro-Oeste
3,4,2,2022,UNIVERSIDADE DE BRASÍLIA,UNB,Universidade,Pública Federal,Pública,Não informado,Não informado,Brasília,DF,Centro-Oeste
4,5,2,2023,UNIVERSIDADE DE BRASÍLIA,UNB,Universidade,Pública Federal,Pública,Não,Não,Brasília,DF,Centro-Oeste


### 3.3 Tabela fato

Primeiro, confirmamos que a chave da fato é única, como o perfilamento verificou (§11). Depois, cada código é trocado pela surrogate key da dimensão correspondente.

A função `buscar_sk` funciona como um PROCV do Excel: para cada linha da fato, procura na dimensão a linha com o mesmo código (ou com o mesmo par código + ano, na `dim_curso` e na `dim_ies`) e devolve a `sk`. Ela gasta bem menos memória do que fazer vários `merge` seguidos numa tabela de quase 2 milhões de linhas.

In [21]:
CHAVE_FATO = ["NU_ANO_CENSO", "CO_CURSO", "CO_MUNICIPIO", "TP_DIMENSAO"]
repetidas = cursos.duplicated(subset=CHAVE_FATO).sum()
print(f"Linhas com a chave da fato repetida: {repetidas}")
assert repetidas == 0

MEDIDAS = [
    "QT_VG_TOTAL", "QT_INSCRITO_TOTAL", "QT_ING", "QT_MAT", "QT_CONC",
    "QT_ING_FIES", "QT_ING_PROUNII", "QT_ING_PROUNIP",
    "QT_ING_PRETA", "QT_ING_PARDA", "QT_ING_INDIGENA",
    "QT_ING_RESERVA_VAGA", "QT_ING_RESERVA_ETNICA",
]


def buscar_sk(fato, dimensao, chaves, sk):
    """Para cada linha da fato, devolve a surrogate key da dimensão. chaves = {coluna na fato: coluna na dimensão}."""
    tabela = dimensao.set_index(list(chaves.values()))[sk]
    if len(chaves) == 1:
        (coluna,) = chaves
        return fato[coluna].astype(tabela.index.dtype).map(tabela)
    indice = pd.MultiIndex.from_frame(fato[list(chaves)])
    return pd.Series(tabela.reindex(indice).to_numpy(), index=fato.index)


BUSCAS = [
    (dim_tempo, {"NU_ANO_CENSO": "ano"}, "sk_tempo"),
    (dim_dimensao, {"TP_DIMENSAO": "co_dimensao"}, "sk_dimensao"),
    (dim_curso, {"CO_CURSO": "co_curso", "NU_ANO_CENSO": "ano"}, "sk_curso"),
    (dim_ies, {"CO_IES": "co_ies", "NU_ANO_CENSO": "ano"}, "sk_ies"),
    (dim_area_cine, {"CO_CINE_ROTULO": "co_cine_rotulo"}, "sk_area_cine"),
    (dim_localidade, {"CO_MUNICIPIO": "co_municipio"}, "sk_localidade"),
]
SKS = [sk for *_, sk in BUSCAS]

fato_oferta_curso = pd.DataFrame(index=cursos.index)
for dimensao, chaves, sk in BUSCAS:
    fato_oferta_curso[sk] = buscar_sk(cursos, dimensao, chaves, sk)
for medida in MEDIDAS:
    fato_oferta_curso[medida.lower()] = cursos[medida]

sem_correspondencia = fato_oferta_curso[SKS].isna().sum()
print("Chaves sem correspondência na dimensão:", sem_correspondencia.to_dict())
assert sem_correspondencia.sum() == 0
fato_oferta_curso[SKS] = fato_oferta_curso[SKS].astype("int64")

print(f"Fato: {len(fato_oferta_curso):,} linhas e {fato_oferta_curso.shape[1]} colunas")
display(fato_oferta_curso.head())

Linhas com a chave da fato repetida: 0
Chaves sem correspondência na dimensão: {'sk_tempo': 0, 'sk_dimensao': 0, 'sk_curso': 0, 'sk_ies': 0, 'sk_area_cine': 0, 'sk_localidade': 0}
Fato: 1,964,978 linhas e 19 colunas


,sk_tempo,sk_dimensao,sk_curso,sk_ies,sk_area_cine,sk_localidade,qt_vg_total,qt_inscrito_total,qt_ing,qt_mat,qt_conc,qt_ing_fies,qt_ing_prounii,qt_ing_prounip,qt_ing_preta,qt_ing_parda,qt_ing_indigena,qt_ing_reserva_vaga,qt_ing_reserva_etnica
0,1,2,137475,551,7,1,0,0,11,8,0,0,0,0,0,1,0,0,0
1,1,3,55078,1,138,3,150,198,0,0,0,0,0,0,0,0,0,0,0
2,1,3,127814,1,186,3,850,552,0,0,0,0,0,0,0,0,0,0,0
3,1,3,44038,1,20,3,117,137,0,0,0,0,0,0,0,0,0,0,0
4,1,3,84921,1,35,3,121,128,0,0,0,0,0,0,0,0,0,0,0


In [22]:
# Conferência: a fato tem as mesmas linhas e totais da referência
conferencia_fato = fato_oferta_curso.groupby("sk_tempo").agg(
    linhas=("qt_ing", "size"), QT_ING=("qt_ing", "sum"), QT_MAT=("qt_mat", "sum"),
)
conferencia_fato.index = conferencia_fato.index.map(dim_tempo.set_index("sk_tempo")["ano"]).rename("ano")

cursos_na_fato = dim_curso[dim_curso["sk_curso"].isin(fato_oferta_curso["sk_curso"])]
conferencia_fato["cursos distintos"] = cursos_na_fato.groupby("ano")["co_curso"].nunique()

conferencia_fato = conferencia_fato[REFERENCIA.columns]
conferencia_fato["confere"] = (conferencia_fato == REFERENCIA).all(axis=1)
display(conferencia_fato)

,linhas,cursos distintos,QT_ING,QT_MAT,confere
ano,,,,,
2022,573019,45296,4756957,9444116,True
2023,671610,46317,4994192,9977217,True
2024,720349,46150,5010613,10227266,True


## 4. Carga no PostgreSQL

Por fim, carregamos as seis dimensões e a fato no PostgreSQL, num schema próprio chamado **`etl`**. O caminho ELT usa outro schema, o que permite comparar os dois resultados no mesmo banco.

### 4.1 Conexão

A senha é pedida na hora com `getpass`, para não ficar escrita no notebook nem ir para o GitHub. No VS Code, uma caixa de texto aparece no topo da janela.

In [23]:
from getpass import getpass
from urllib.parse import quote_plus

from sqlalchemy import create_engine, text

USUARIO = "postgres"
HOST = "localhost"
PORTA = 5432
BANCO = "censo_superior"
SCHEMA = "etl"

senha = getpass("Senha do PostgreSQL: ")
engine = create_engine(f"postgresql+psycopg2://{USUARIO}:{quote_plus(senha)}@{HOST}:{PORTA}/{BANCO}")

with engine.connect() as conexao:
    print("Conectado:", conexao.execute(text("SELECT version()")).scalar())

Conectado: PostgreSQL 18.7 on x86_64-windows, compiled by msvc-19.44.35229, 64-bit


### 4.2 Carga das tabelas

O schema `etl` é apagado e recriado a cada execução, para que o notebook possa ser rodado de novo do zero sem duplicar dados. As dimensões são carregadas antes da fato.

A fato tem quase 2 milhões de linhas. Com o `to_sql` padrão, o pandas envia os dados linha a linha como comandos `INSERT`, o que levaria muitos minutos. Por isso a carga é feita em dois passos, pela função `carregar_tabela`:

1. o `to_sql` com **zero linhas** (`df.head(0)`) cria a tabela vazia, com os tipos de coluna corretos;
2. a tabela inteira é convertida para CSV em memória (`to_csv`) e enviada com o comando **`COPY`** do PostgreSQL, feito para cargas em massa.

In [24]:
import time
from io import StringIO


def carregar_tabela(df, nome):
    """Cria a tabela no schema e carrega os dados com o COPY do PostgreSQL."""
    df.head(0).to_sql(nome, engine, schema=SCHEMA, index=False)
    buffer = StringIO()
    df.to_csv(buffer, index=False, header=False)
    buffer.seek(0)
    with engine.begin() as conexao:
        with conexao.connection.cursor() as cursor:
            cursor.copy_expert(f"COPY {SCHEMA}.{nome} FROM STDIN WITH CSV", buffer)


with engine.begin() as conexao:
    conexao.execute(text(f"DROP SCHEMA IF EXISTS {SCHEMA} CASCADE"))
    conexao.execute(text(f"CREATE SCHEMA {SCHEMA}"))

TABELAS = {**DIMENSOES, "fato_oferta_curso": fato_oferta_curso}

for nome, df in TABELAS.items():
    inicio = time.time()
    carregar_tabela(df, nome)
    print(f"{nome}: {len(df):,} linhas carregadas em {time.time() - inicio:.1f} s")

dim_tempo: 3 linhas carregadas em 0.1 s
dim_dimensao: 4 linhas carregadas em 0.0 s
dim_curso: 137,763 linhas carregadas em 0.4 s
dim_ies: 7,736 linhas carregadas em 0.0 s
dim_area_cine: 372 linhas carregadas em 0.0 s
dim_localidade: 3,681 linhas carregadas em 0.0 s
fato_oferta_curso: 1,964,978 linhas carregadas em 20.4 s


### 4.3 Chaves primárias e estrangeiras

O `to_sql` cria as tabelas, mas não as chaves. Adicionamos a chave primária de cada dimensão e as chaves estrangeiras da fato. Assim o próprio banco garante que toda linha da fato aponta para uma linha existente em cada dimensão.

In [25]:
with engine.begin() as conexao:
    for nome, df in DIMENSOES.items():
        sk = df.columns[0]
        conexao.execute(text(f"ALTER TABLE {SCHEMA}.{nome} ADD PRIMARY KEY ({sk})"))
        conexao.execute(text(
            f"ALTER TABLE {SCHEMA}.fato_oferta_curso "
            f"ADD CONSTRAINT fk_fato_{nome} FOREIGN KEY ({sk}) REFERENCES {SCHEMA}.{nome} ({sk})"
        ))
print("Chaves primárias e estrangeiras criadas")

Chaves primárias e estrangeiras criadas


### 4.4 Verificação no banco

Conferimos, direto no PostgreSQL, a quantidade de linhas de cada tabela e, por ano, as linhas, os cursos distintos e os totais de ingressantes e matrículas. Os cursos são contados com `COUNT(DISTINCT co_curso)`, e não pelo número de linhas, porque um curso EAD aparece uma vez por polo (perfilamento §16).

In [26]:
contagem = " UNION ALL ".join(
    f"SELECT '{nome}' AS tabela, COUNT(*) AS linhas FROM {SCHEMA}.{nome}" for nome in TABELAS
)
display(pd.read_sql(contagem, engine))

,tabela,linhas
0,dim_tempo,3
1,dim_dimensao,4
2,dim_curso,137763
3,dim_ies,7736
4,dim_area_cine,372
5,dim_localidade,3681
6,fato_oferta_curso,1964978


In [27]:
consulta = f"""
SELECT t.ano,
       COUNT(*)                   AS linhas,
       COUNT(DISTINCT c.co_curso) AS cursos_distintos,
       SUM(f.qt_ing)::bigint      AS ingressantes,
       SUM(f.qt_mat)::bigint      AS matriculas
FROM {SCHEMA}.fato_oferta_curso AS f
JOIN {SCHEMA}.dim_tempo AS t ON t.sk_tempo = f.sk_tempo
JOIN {SCHEMA}.dim_curso AS c ON c.sk_curso = f.sk_curso
GROUP BY t.ano
ORDER BY t.ano
"""
no_banco = pd.read_sql(consulta, engine).set_index("ano")
no_banco["confere com a referência"] = (
    no_banco[["linhas", "cursos_distintos", "ingressantes", "matriculas"]].values == REFERENCIA.values
).all(axis=1)
display(no_banco)

,linhas,cursos_distintos,ingressantes,matriculas,confere com a referência
ano,,,,,
2022,573019,45296,4756957,9444116,True
2023,671610,46317,4994192,9977217,True
2024,720349,46150,5010613,10227266,True


In [28]:
consulta = f"""
SELECT t.ano,
       d.modalidade,
       SUM(f.qt_vg_total)::bigint AS vagas,
       SUM(f.qt_ing)::bigint      AS ingressantes,
       SUM(f.qt_mat)::bigint      AS matriculas
FROM {SCHEMA}.fato_oferta_curso AS f
JOIN {SCHEMA}.dim_tempo    AS t ON t.sk_tempo = f.sk_tempo
JOIN {SCHEMA}.dim_dimensao AS d ON d.sk_dimensao = f.sk_dimensao
GROUP BY t.ano, d.modalidade
ORDER BY t.ano, d.modalidade
"""
display(pd.read_sql(consulta, engine))

,ano,modalidade,vagas,ingressantes,matriculas
0,2022,EAD,17171895,3100556,4330934
1,2022,Presencial,5658590,1656401,5113182
2,2023,EAD,19180966,3314402,4913281
3,2023,Presencial,5440999,1679790,5063936
4,2024,EAD,18590273,3347573,5189391
5,2024,Presencial,5075146,1663040,5037875


O pipeline ETL está completo: os microdados de 2022, 2023 e 2024 foram extraídos, transformados em memória e carregados como um esquema estrela no schema `etl`, com as linhas, os cursos distintos e os totais conferidos contra a referência do perfilamento. Os mesmos números servem para a conferência com o caminho ELT.